# Les arbres de Paris 

## Une analyse de la population d'arbres dans Paris 

Pour ce projet nous avons décidé de travailler sur les arbres dans Paris. Pour cela nous nous appuyons sur la base de données établie par la mairie de Paris recensant les arbres : leur emplacement, taille, circonférence ou encore espèce. Nous utilisons aussi la base de données sur les IRIS en france afin d'établir des liens géographique par iris. 

### Nettoyage de la base de données

Nous commencons par charger et nettoyer les bases de données 

On remarque que notre table contient trop de colonnes inutiles : 

- IDBASE : id de la ligne inutile
- cleabs : Identifiant technique IGN
- IDEMPLACEMENT, COMPLEMENT ADRESSE, LIEU / ADRESSE : indications géographique inutiles comme on possède la localisation. 
- iris : déjà contenu dans code_iris, qui est un indentifiant complet et unique. 
- geo_point_2d, lat et lon : redondant avec geometry
- 'nom_commune : variable doublée (arrondissement)
- index_right : rajoutée avec la jointure
- TYPE EMPLACEMENT : modalité Arbre partout. 

In [ ]:
import sys
sys.path.append('../src')
import utils
import pandas as pd
import s3fs
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from matplotlib.lines import Line2D


In [ ]:
arbres = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/60433484-f30e-44ef-a362-e5553a9b7a42", sep = ";")
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "raphcrre"
PATH_IRIS = f"{MY_BUCKET}/diffusion/projet_arbres/iris.gpkg"

with fs.open(PATH_IRIS, 'rb') as f:
    iris_france = gpd.read_file(f)

iris = iris_france[iris_france['code_insee'].astype(str).str.startswith('75')].copy()
df_arbres = utils.jointure_arbres_iris(arbres, iris)
df_arbres

df_arbres = utils.suppression_colonnes(df_arbres)


df_arbres = utils.nettoyer_valeurs_aberrantes(df_arbres)

print("-" * 30)
print(f"Statistiques après nettoyage :")
print(f"Hauteur moyenne : {df_arbres['hauteur_m'].mean():.2f} m")
print(f"Circonférence moyenne : {df_arbres['circonference_cm'].mean():.2f} cm")
print("-" * 30)
df_arbres


## Visualisation

### TOP 10 des essences (types d'arbres à Paris)
Nous voulons maintenant faire quelques statistiques descriptives sur les arbres à Paris. 
D'abord nous nous demandons quelles sont les epsèces d'arbres les plus fréquentes. 

In [ ]:
df_top = utils.get_top_species(arbres, column_name = "LIBELLE FRANCAIS")


plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    data=df_top, 
    x='Nombre', 
    y='Espece', 
    palette='viridis'
)

plt.title('Top 10 des essences d\'arbres à Paris', fontsize=15, pad=20)
plt.xlabel('Nombre d\'arbres')
plt.ylabel('Essence (nom commun)')

for i, p in enumerate(ax.patches):
    percentage = f"{df_top['Pourcentage'].iloc[i]}%"
    ax.annotate(percentage, (p.get_width(), p.get_y() + p.get_height()/2),
                ha='left', va='center', xytext=(5, 0), textcoords='offset points')

plt.show()

#### Répartition du stade de développement 

Puis nous nous interrogeons sur l'ancienneté des arbres. 

In [ ]:
df_stade = utils.get_development_stats(arbres)

plt.figure(figsize=(8, 8))
colors = ['#66b3ff', '#99ff99', '#ffcc99', '#ff9999'] # Couleurs douces

plt.pie(
    df_stade['Proportion'], 
    labels=df_stade['Stade'], 
    autopct='%1.1f%%', 
    startangle=140, 
    colors=colors,
    pctdistance=0.85
)

centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title("Répartition des stades de développement des arbres à Paris", fontsize=14)
plt.tight_layout()
plt.show()

### Modélisation des stades de développement 

On peut ensuite modéliser ces stades moyens de développement par Iris ou par arrondissement de façon intéractive pour se rendre davatage compte de la typologie parisienne des arbres. 

In [ ]:
print("\n--- INFOS GÉNÉRALES ---")

gdf_iris_stats = utils.preparer_geodata_stade(iris, df_arbres, niveau='iris')
gdf_iris_stats[['code_iris', 'nom_iris', 'stade_dominant', 'pct_Jeune', 'pct_Adulte', 'pct_Vieux']].head()

mapping_arrdt = {
    '75101': 'PARIS 1ER ARRDT',
    '75102': 'PARIS 2E ARRDT',
    '75103': 'PARIS 3E ARRDT',
    '75104': 'PARIS 4E ARRDT',
    '75105': 'PARIS 5E ARRDT',
    '75106': 'PARIS 6E ARRDT',
    '75107': 'PARIS 7E ARRDT',
    '75108': 'PARIS 8E ARRDT',
    '75109': 'PARIS 9E ARRDT',
    '75110': 'PARIS 10E ARRDT',
    '75111': 'PARIS 11E ARRDT',
    '75112': 'PARIS 12E ARRDT',
    '75113': 'PARIS 13E ARRDT',
    '75114': 'PARIS 14E ARRDT',
    '75115': 'PARIS 15E ARRDT',
    '75116': 'PARIS 16E ARRDT',
    '75117': 'PARIS 17E ARRDT',
    '75118': 'PARIS 18E ARRDT',
    '75119': 'PARIS 19E ARRDT',
    '75120': 'PARIS 20E ARRDT',
}

iris_avec_arrdt = iris.copy()
iris_avec_arrdt['arrondissement'] = iris_avec_arrdt['code_insee'].map(mapping_arrdt)

iris_avec_arrdt = iris_avec_arrdt[iris_avec_arrdt['arrondissement'].notna()]

gdf_arrdt = iris_avec_arrdt.dissolve(by='arrondissement').reset_index()[['arrondissement', 'geometry']]

gdf_arrdt_stats = utils.preparer_geodata_stade(gdf_arrdt, df_arbres, niveau='arrondissement')

print(gdf_arrdt_stats[['arrondissement', 'stade_dominant', 'total', 'pct_Jeune', 'pct_Adulte', 'pct_Vieux']])

carte = utils.carte_stade_paris(
    gdf_iris_stats=gdf_iris_stats,
    gdf_arrdt_stats=gdf_arrdt_stats,
    titre="Stade dominant des arbres à Paris"
)
carte

Les 20 IRIS vides se répartissent en 3 catégories bien distinctes :

Type A (7 IRIS) → zones d'activités commerciales/tertiaires (Palais Royal, Madeleine, Chaussée d'Antin...). Pas d'arbres municipaux recensés, ce qui est logique.
Type D (1 IRIS) → Tuileries, c'est une zone de grande emprise (le jardin des Tuileries). Paradoxalement sans données, probablement parce que les arbres des jardins historiques gérés par l'État ne sont pas dans le dataset municipal de Paris.
Type H (12 IRIS) → zones à forte emprise d'équipements publics (hôpitaux, casernes, universités...). Même raison : les arbres de ces emprises ne sont pas gérés par la ville de Paris donc absents du dataset.

C'est une limite connue du dataset arbres.data.gouv.fr qui ne recense que les arbres gérés par la Direction des Espaces Verts de la Ville de Paris, et exclut donc les arbres appartenant à l'État, aux hôpitaux, aux universités, etc. 

### Modélisation de la densité d'arbres à Paris

On veut ensuite représenter la densité de population d'arbres dans Paris. Selon les IRIS ou les arrondissement et toujours de manière intéractive. On ajoute aussi un figuré ponctuel avec les arbres remarquables de Paris.

In [ ]:
gdf_iris_densite = utils.calculer_densite_par_zone(df_arbres, iris, groupby_col='code_iris')

gdf_arrdt_densite = utils.calculer_densite_par_zone(df_arbres, gdf_arrdt, groupby_col='arrondissement')

gdf_remarquables = utils.extraire_arbres_remarquables(df_arbres, top_n_hauteur=50, top_n_circonf=50)
print(f"{len(gdf_remarquables)} arbres remarquables extraits")
print(gdf_remarquables[['libelle_francais', 'hauteur_m', 'circonference_cm', 'motif_remarquable']].head())

carte_densite = utils.carte_densite_paris(
    gdf_iris_densite=gdf_iris_densite,
    gdf_arrdt_densite=gdf_arrdt_densite,
    gdf_remarquables=gdf_remarquables,
    titre="Densité d'arbres à Paris"
)
carte_densite


## Clustering 

Nous utilisons maintenant une méthode de clustering pour établir des profils types d'iris en fonction de leur population d'arbres. 

Préaration de la base de données pour le clustering + test du coude pour trouver nombre de cluster idéal 

In [ ]:
donnees_clustering = utils.preparer_clustring(df_arbres)

utils.choix_k_coude(donnees_clustering)

iris_clustered = utils.applique_clustering(donnees_clustering, n_clusters=4)


D'après le théorème du coude, il semble plus optimal de choisir un k = 4
On peut donc faire notre clustering avec 4 clusters

In [ ]:
custom_palette = {
    2: "#a1d99b",  
    0: "#41ab5d",  
    1: "#e5f5e0",  
    3: "#00441b"   
}

utils.plot_morpho(iris_clustered, custom_palette)

stats = utils.moyennes_cluster(iris_clustered)
print(stats)


En affichant les moyennes de hauteurs, circonferences et nombre par cluster, on voit bien que 4 groupes bien distincts se créeent: 

1.	Cluster 3 (Le plus foncé) : Les "Forêts Urbaines". Moyenne de 2832 arbres par Iris et une hauteur max record de 30m. C'est le cluster des bois.
2.	Cluster 0 (Vert soutenu) : Les "Quartiers Arborés et Anciens". Ils ont le nombre d'arbres le plus élevé après les bois (201) et la circonférence moyenne la plus forte (102 cm).
3.	Cluster 2 (Vert moyen) : Le "Standard Parisien". Un profil équilibré, environ 140 arbres par IRIS avec une morphologie moyenne.
4.	Cluster 1 (Vert très clair) : Les "Zones Jeunes ou Minérales". Plus petites circonférences (57 cm) et moins d'arbres.

On peut maintenant représenter nos clusters sur une carte: 

In [ ]:
couleurs_vertes = {
    "1.0": "#e5f5e0",       
    "2.0": "#a1d99b",       
    "0.0": "#41ab5d",       
    "3.0": "#00441b",       
    "Sans arbres": "#D3D3D3" 
}


map_df = iris.merge(iris_clustered[['code_iris', 'cluster']], on='code_iris', how='left')

map_df['cluster'] = map_df['cluster'].fillna("Sans arbres")

utils.plot_map_cluster(map_df, couleurs_vertes)

